# Gesture evaluation

Run from the repository root or the notebooks directory. Use consented clips for human-performance claims. Synthetic fixture runs only validate the pipeline.

In [ ]:
import json
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from handgest import LABELS
from handgest.dataset import load
from handgest.features import to_features
from handgest.predict import load_predictor
from handgest.report import evaluate

root = Path.cwd() if Path("configs").exists() else Path.cwd().parent
clips = load(root / "data/processed")
print("Clips by label:", Counter(c.label for c in clips))
print("Clips by subject:", Counter(c.subject for c in clips))
print("Mean detected-frame fraction:", np.mean([c.mask.mean() for c in clips]))
log = root / "data/processed/extraction.csv"
if log.exists():
    print("\n".join(row for row in log.read_text().splitlines() if "dropped" in row)[:1500])

In [ ]:
reports = {}
for model in ("torch-tcn", "torch-gru", "jax-tcn"):
    report = root / "runs" / model / "report.json"
    if report.exists():
        reports[model] = json.loads(report.read_text())
baseline = root / "runs/baseline.json"
if baseline.exists():
    reports.update(json.loads(baseline.read_text()))
for name, report in reports.items():
    print(
        name,
        report["summary"],
        "parameters:",
        report["folds"][0].get("parameters"),
        "mean step ms:",
        [f.get("mean_step_ms") for f in report["folds"]],
    )

In [ ]:
report = reports["torch-tcn"]["folds"][0]
subject = report["test_subject"]
held_out = [clip for clip in clips if clip.subject == subject]
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(report["confusion"])
ax.set(
    xticks=range(8),
    xticklabels=LABELS,
    yticks=range(8),
    yticklabels=LABELS,
    xlabel="Prediction",
    ylabel="Actual",
)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
for label in LABELS:
    print(label, report["per_class"][label])

## Robustness and latency

Use a fixed observation window for speed changes. Faster gestures finish early and hold the final frame. Slower gestures may leave the gesture incomplete. This tests observation latency rather than reparameterizing the entire clip.

In [ ]:
predictor = load_predictor(root / "runs/torch-tcn/fold-0/best.onnx")
expected = [LABELS.index(c.label) for c in held_out]


def run(noise=0.0, drop=0.0, speed=1.0):
    rng = np.random.default_rng(13)
    predictions, feature_ms, inference_ms = [], [], []
    for clip in held_out:
        indices = np.minimum(
            (np.arange(len(clip.points)) * speed).astype(int), len(clip.points) - 1
        )
        points = clip.points[indices].copy()
        points += rng.normal(0, noise, size=points.shape)
        mask = clip.mask[indices] & (rng.random(len(points)) >= drop)
        start = time.perf_counter()
        x = to_features(points, mask)
        feature_ms.append((time.perf_counter() - start) * 1000)
        start = time.perf_counter()
        if np.any(x):
            predictor.logits(x[None])
        inference_ms.append((time.perf_counter() - start) * 1000)
        predicted = predictor.predict(points, mask)["label"]
        predictions.append(LABELS.index(predicted))
    return predictions, np.median(feature_ms), np.median(inference_ms)


settings = {
    "clean": {},
    "noise": {"noise": 0.01},
    **{f"drop {r:.0%}": {"drop": r} for r in (0.1, 0.2, 0.3)},
    "0.7x": {"speed": 0.7},
    "1.3x": {"speed": 1.3},
}
results = {name: run(**args) for name, args in settings.items()}
plt.bar(results, [evaluate(expected, row[0])["macro_f1"] for row in results.values()])
plt.ylabel("Macro F1")
plt.show()
clean, feature_ms, inference_ms = results["clean"]
print("CPU median feature/clip ms:", feature_ms, "ONNX/clip ms:", inference_ms)

In [ ]:
errors = [(clip, label) for clip, label in zip(held_out, clean) if clip.label != LABELS[label]]
for clip, label in errors[:4]:
    x = to_features(clip.points, clip.mask)
    plt.plot(x[:, 63], x[:, 64], marker=".")
    plt.title(f"Actual {clip.label} | Predicted {LABELS[label]} | Detected {clip.mask.mean():.0%}")
    plt.xlabel("Wrist displacement X / palm size")
    plt.ylabel("Wrist displacement Y / palm size")
    plt.show()
print("Review the shown trajectories before attributing errors to shape, motion or tracking loss.")